# 4. LCEL (LangChain Expression Language)

LCEL은 프롬프트, LLM, 출력 파서 같은 컴포넌트를 **파이프(`|`) 연산자로 선언적으로 연결**하는 LangChain 1.x의 표준 체인 구성 방식입니다.

| 항목 | 설명 |
|---|---|
| 정의 | 체인을 함수형 방식으로 구성하는 표준 방법 |
| 핵심 메커니즘 | 모든 컴포넌트를 Runnable 인터페이스로 통일하고, 파이프 연산자로 앞 요소의 출력을 뒤 요소의 입력으로 연결 |
| 목적 | LLM 애플리케이션의 모듈화, 확장성, 유지보수 편의성 향상 |

## 학습 목표

1. LCEL의 파이프 연결 구조를 이해합니다.
2. `prompt | llm | parser` 체인을 만들고 4가지 실행 방법으로 실행합니다.
3. `RunnableLambda`, `RunnableParallel`, `RunnablePassthrough`를 사용합니다.

**기준 버전**: LangChain 1.x, `langchain-openai` 1.x, 모델 `gpt-6-luna`

## 1. LCEL의 핵심 개념

- **선언적 연결**: 컴포넌트를 `|`로 연결해 데이터 흐름을 직관적으로 표현합니다.
  ```python
  chain = prompt | llm | output_parser
  ```
- **간결한 코드**: 예전 객체 방식보다 짧고 읽기 쉽습니다.
  ```python
  # 예전 방식 (LangChain 1.x에서 제거됨)
  chain = LLMChain(llm=llm, prompt=prompt)
  # LCEL 방식
  chain = prompt | llm
  ```
- **모듈성·확장성**: 컴포넌트를 조합해 복잡한 애플리케이션을 만들고, 일부만 바꿔 끼우기 쉽습니다.

### LCEL이 강력한 이유

LCEL로 만든 체인은 별도 코드 없이 다음 기능을 **자동으로** 지원합니다.

- **스트리밍**: 토큰이 생성되는 즉시 출력해 사용자 경험(UX)을 높입니다.
- **비동기**: 웹 서버처럼 동시 요청이 많은 환경에서 효율적으로 처리합니다.
- **병렬 실행**: 여러 LLM 호출이나 도구 실행을 동시에 처리해 시간을 줄입니다.
- **폴백(Fallback)**: 특정 단계에서 오류가 나면 대체 모델이나 경로를 지정할 수 있습니다.

### Runnable 인터페이스의 주요 메서드

| 메서드 | 실행 방식 | 설명 |
|---|---|---|
| `.invoke()` | 동기 | 입력 1개로 체인을 실행하고 결과 반환 |
| `.ainvoke()` | 비동기 | async/await 환경에서 실행 |
| `.batch()` | 배치 | 여러 입력을 리스트로 받아 병렬 처리 |
| `.stream()` | 스트리밍 | 결과 청크를 생성 즉시 반환 |

### 실습 환경 준비

터미널에서 필요한 패키지를 설치하고, 프로젝트 최상위 폴더의 `.env` 파일에 API 키와 모델명을 저장합니다.

```text
OPENAI_API_KEY="YOUR_OPENAI_API_KEY"
OPENAI_DEFAULT_MODEL="gpt-6-luna"
```

> API 키는 코드 셀이나 GitHub에 직접 입력하지 마세요.

```bash
uv add langchain langchain-openai python-dotenv
```

In [1]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

default_model = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")

# reasoning_effort="none": 수업 예제는 빠른 응답과 낮은 비용을 위해 추론 기능을 끕니다.
llm = ChatOpenAI(model=default_model, reasoning_effort="none")
print("사용 모델:", default_model)

사용 모델: gpt-6-luna


> **gpt-6-luna 사용 시 참고**
> - `reasoning_effort`는 `none` / `low` / `medium`(기본) / `high` / `xhigh` / `max`를 지원합니다. 강의 예제는 `none`으로 통일합니다.
> - `temperature`는 추론이 꺼져 있을 때(`reasoning_effort="none"`)만 적용됩니다.
> - 모델명·요금·지원 파라미터는 바뀔 수 있으므로 수업 전에 [OpenAI 모델 문서](https://developers.openai.com/api/docs/models/gpt-6-luna)를 확인하세요.

## 2. Runnable Chain 만들기

### 2-1. prompt | llm

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("{city}에서 가장 유명한 랜드마크는 무엇인가요?")

chain = prompt | llm
response = chain.invoke({"city": "파리"})
print(response)          # AIMessage 객체 전체

| 구성 요소 | 역할 | 비유 |
|---|---|---|
| `PromptTemplate` | 입력값을 넣을 빈칸이 있는 질문 틀 | 레시피/양식 |
| `ChatOpenAI` | 질문을 받아 답변하는 AI 엔진 | 요리사 |
| 파이프 연산자 (`\|`) | 구성 요소를 순서대로 잇는 연결고리 | 컨베이어 벨트 |
| `invoke()` | 완성된 체인에 데이터를 넣고 실행 | 시동 버튼 |

### 2-2. prompt | llm | StrOutputParser

`StrOutputParser`는 응답 객체(`AIMessage`)에서 **텍스트(content)만 꺼내 문자열로 변환**합니다.

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template("'{topic}' 주제에 대해 한 문장으로 설명해 줘.")
parser = StrOutputParser()

chain = prompt | llm | parser   # 프롬프트 → LLM → 문자열

result = chain.invoke({"topic": "LangChain"})
print(isinstance(result, str))   # True: 문자열로 변환됨
print(result)

True
LangChain은 대규모 언어 모델을 외부 데이터와 도구에 연결해 복잡한 AI 애플리케이션을 구축하도록 돕는 프레임워크입니다.


### 2-3. 같은 체인을 4가지 방식으로 실행하기

In [3]:
# 1. 단일 동기 실행
print(chain.invoke({"topic": "인공지능"}))

# 2. 비동기 실행
print(await chain.ainvoke({"topic": "인공지능"}))

# 3. 여러 입력 동시 처리 (배치)
print(chain.batch([{"topic": "딥러닝"}, {"topic": "머신러닝"}]))

# 4. 실시간 스트리밍
for chunk in chain.stream({"topic": "생성형 AI"}):
    print(chunk, end="", flush=True)

인공지능은 컴퓨터가 학습과 추론을 통해 인간의 지능이 필요한 작업을 수행하도록 만드는 기술입니다.
인공지능은 컴퓨터가 학습과 추론을 통해 사람처럼 문제를 해결하고 결정을 내리도록 하는 기술입니다.
['딥러닝은 여러 층으로 이루어진 인공신경망이 데이터를 학습해 패턴을 찾아 예측이나 분류를 수행하는 기술입니다.', '머신러닝은 컴퓨터가 데이터를 통해 패턴을 학습하고, 이를 바탕으로 예측이나 결정을 내리도록 하는 기술입니다.']
생성형 AI는 학습한 데이터를 바탕으로 글, 이미지, 음악, 코드 등 새로운 콘텐츠를 만들어 내는 인공지능입니다.

## 3. LCEL의 모든 체인 요소는 Runnable이다

### 3-1. RunnableLambda

- 일반 **파이썬 함수를 Runnable로 감싸** 체인 중간에 끼워 넣게 해 주는 어댑터입니다.
- 일반 함수는 `|` 연산자를 쓸 수 없으므로, 아래 같은 커스텀 처리가 필요할 때 사용합니다.
  - LLM 입력 전 텍스트 전처리 (공백 제거, 단어 필터링 등)
  - 출력 결과를 서비스 형태(JSON, HTML 등)로 재가공
  - 외부 API나 DB 조회 결과를 중간에 삽입

가장 단순한 예시부터 보겠습니다.

In [4]:
from langchain_core.runnables import RunnableLambda

def greet(name):
    return f"Hello, {name}!"

greet_runnable = RunnableLambda(greet)   # 함수를 Runnable로 감싸기
print(greet_runnable.invoke("Alice"))   # Hello, Alice!

Hello, Alice!


#### 파이썬 함수만으로 파이프라인 만들기

결과는 `mul_two(add_one(3))`과 같습니다.

In [ ]:
from langchain_core.runnables import RunnableLambda

def add_one(x):
    return x + 1

def mul_two(x):
    return x * 2

chain = RunnableLambda(add_one) | RunnableLambda(mul_two)
print(chain.invoke(3))   # 3+1=4 → 4*2=8

#### 함수 → LLM으로 연결하기

- `|` 연산자는 **이전 단계의 출력을 다음 단계의 입력**으로 자동 연결합니다.
- LLM은 문자열(또는 메시지)을 입력으로 받으므로, 숫자 결과를 문장으로 바꾼 뒤 LLM에 전달합니다.
- `invoke()`의 입력 형태는 **체인의 첫 번째 Runnable**이 결정합니다.

| 첫 Runnable | invoke 입력 |
|---|---|
| `RunnableLambda` | 함수가 받는 어떤 타입이든 가능 |
| `PromptTemplate` / `ChatPromptTemplate` | `dict` |
| `ChatOpenAI` | 문자열 또는 메시지 리스트 |

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = (
    RunnableLambda(add_one)
    | RunnableLambda(mul_two)
    | RunnableLambda(lambda x: f"숫자 {x}에 대해 재미있는 사실을 한 문장으로 알려 줘")
    | llm
    | StrOutputParser()
)
print(chain.invoke(3))

### 실습

- 오늘 날짜를 구하는 Runnable(`date_runnable`)을 만드세요.
- 질문: "오늘 점심 뭐 먹을까?"
- 체인: `date_runnable | prompt | llm`
  - `date_runnable` → 오늘 날짜와 질문을 담은 딕셔너리 반환
  - `prompt` → 날짜가 포함된 질문 완성
  - `llm` → 답변 생성

In [ ]:
# 여기에 실습 코드를 작성하세요.

#### 정답 예시

In [7]:
from datetime import datetime
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 오늘 날짜와 질문을 딕셔너리로 반환하는 Runnable
date_runnable = RunnableLambda(
    lambda question: {
        "today": datetime.now().strftime("%Y-%m-%d"),
        "question": question,
    }
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 음식 추천 어시스턴트입니다.날짜는 꼭 알려주세요"),
    ("human", "오늘은 {today}입니다. {question}"),
])

# LLM 설정
llm = ChatOpenAI(model=default_model)

chain = date_runnable | prompt | llm | StrOutputParser()
print(chain.invoke("오늘 점심 뭐 먹을까?"))

오늘은 **2026년 10월 10일**이에요. 🍂  
점심으로는 따뜻하고 든든한 **버섯불고기 덮밥** 어때요? 가을 버섯 향도 좋고 한 그릇으로 간편하게 먹기 좋아요.  
조금 가볍게 먹고 싶다면 **들깨버섯칼국수**도 추천해요!


In [6]:
prompt.invoke(date_runnable.invoke("오늘 점심 뭐 먹을까?")).to_string()

'System: 당신은 친절한 음식 추천 어시스턴트입니다.\nHuman: 오늘은 2026-10-10입니다. 오늘 점심 뭐 먹을까?'

### 3-2. RunnableParallel

- **같은 입력**을 여러 체인에 **동시에** 보내고, 결과를 **하나의 딕셔너리**로 모읍니다.
- 지금까지의 체인(`prompt | llm | parser`)이 순차 구조였다면, `RunnableParallel`은 서로 독립적인 작업을 한 번에 처리할 때 씁니다.

```text
            ┌─ pros_chain ─┐
{"topic"} ──┤              ├──> {"pros": "...", "cons": "..."}
            └─ cons_chain ─┘
```

In [8]:
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

pros_chain = ChatPromptTemplate.from_template("{topic}의 장점을 한 문장으로 알려 줘") | llm | parser
cons_chain = ChatPromptTemplate.from_template("{topic}의 단점을 한 문장으로 알려 줘") | llm | parser

parallel_chain = RunnableParallel(pros=pros_chain, cons=cons_chain)

result = parallel_chain.invoke({"topic": "전기차"})
print(result)
# 출력 예: {'pros': '전기차는 ...', 'cons': '전기차는 ...'}

{'pros': '전기차는 주행 중 배출가스가 없고 연료비와 소음이 적어 더 친환경적이고 경제적입니다.', 'cons': '전기차는 충전 시간이 길고 충전소가 부족한 곳에서는 이용이 불편할 수 있습니다.'}


In [13]:
print(result.keys())
print(result['pros'])
print(result['cons'])

dict_keys(['pros', 'cons'])
전기차는 주행 중 배출가스가 없고 연료비와 소음이 적어 더 친환경적이고 경제적입니다.
전기차는 충전 시간이 길고 충전소가 부족한 곳에서는 이용이 불편할 수 있습니다.


In [15]:
summary_prompt = ChatPromptTemplate.from_template(
    "장점: {pros}\n단점: {cons}\n\n위 내용을 종합해서 결론을 내려줘."
)

full_chain = parallel_chain | summary_prompt | llm | StrOutputParser()
print(full_chain.invoke("전기차를 사야할까?"))

종합하면, 전기차는 연료비와 정비 부담이 적고 주행 중 배출가스가 없어 경제성과 친환경성을 갖춘 선택입니다. 다만 충전 인프라와 충전 시간이 여전히 제약이 될 수 있으므로, **평소 충전이 편하고 주행 거리가 주로 짧거나 일정한 사람에게 특히 적합합니다.** 장거리 운행이 잦거나 충전 시설이 부족한 지역에 산다면 구매 전에 충전 여건을 충분히 고려하는 것이 좋습니다.


### 3-3. RunnablePassthrough

입력을 받아 **아무 가공 없이 그대로** 출력합니다.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough()
print(passthrough.invoke("안녕하세요"))   # 안녕하세요

`RunnableParallel`과 함께 쓰면 "이 자리에는 원래 입력값을 그대로 넣어 줘"라는 표시로 사용됩니다.

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

chain = RunnableParallel(
    original=RunnablePassthrough(),   # 입력 그대로
    upper=lambda x: x.upper(),        # 입력을 대문자로
)

print(chain.invoke("hello"))
# 출력: {'original': 'hello', 'upper': 'HELLO'}